In [7]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.applications import MobileNetV2
from keras import Sequential
from keras.utils import image_dataset_from_directory
from keras.layers import Dense,BatchNormalization,Dropout,GlobalAveragePooling2D
from keras.regularizers import L2
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix

In [4]:
test_path = r"C:\Users\user\OneDrive - REVA University\Projects\BrainVision-AI\Dataset\classification_task\test"
train_path = r"C:\Users\user\OneDrive - REVA University\Projects\BrainVision-AI\Dataset\classification_task\train"

In [5]:
BATCH_SIZE = 32
EPOCH = 30
SIZE = (224,224)

In [14]:
train_datagen = ImageDataGenerator(rescale=1/255.0)
val_datagen = ImageDataGenerator(rescale=1/255.0)
test_datagen = ImageDataGenerator(rescale=1/255.0)

In [16]:
train_dataset = image_dataset_from_directory(
    train_path,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=SIZE,
    batch_size=BATCH_SIZE,
)
val_dataset = image_dataset_from_directory(
    train_path,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=SIZE,
    batch_size=BATCH_SIZE
)

Found 5000 files belonging to 4 classes.
Using 4000 files for training.
Found 5000 files belonging to 4 classes.
Using 1000 files for validation.


In [20]:
class_names = train_dataset.class_names
num_classes = len(class_names)

In [17]:
test_dataset = image_dataset_from_directory(
    test_path,
    image_size=SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

Found 1000 files belonging to 4 classes.


In [18]:
base_model = MobileNetV2(input_shape=(224,224,3),include_top=False,weights="imagenet")
base_model.trainable = False

In [21]:
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dropout(0.2),
    Dense(64,activation="relu"),
    Dense(num_classes, activation="softmax")
])

In [28]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [29]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    mode="min",
    restore_best_weights=True,
    verbose=1
)

In [30]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCH,
    callbacks=[early_stopping]
)

Epoch 1/30
 37/125 ━━━━━━━━━━━━━━━━━━━━ 45s 520ms/step - accuracy: 0.5818 - loss: 0.9503

KeyboardInterrupt: 

In [ ]:
os.makedirs("Models", exist_ok=True)

model.save("Models/brain_vision_ai.keras")